# Qwen-Image-2.1 Standalone Runner & Segmentation-Aware Editor (Google Colab)

Run **Base Qwen-Image-2.1 (30–50 steps, CFG 3.5–5.0, negative prompts)** and **Viggle Turbo (4 steps)** on Google Colab (Free T4 or L4/A100 GPU) without ComfyUI.

### Capabilities:
1. **Text-to-Image Generation** (Photorealistic 1024×1024 synthesis)
2. **Global Image Editing** (Semantic image-to-image)
3. **Segmentation-Aware Pixel-Preserving Editing** (Zero-drift exact preservation outside edit bounds)

In [ ]:
# 1. Verify NVIDIA GPU (Supports T4, L4, V100, A100)
!nvidia-smi

In [ ]:
# 2. Clone Repository, Install Dependencies & Build CUDA Engine
import os
if not os.path.exists("vision-model"):
    !git clone https://github.com/samirpatil2000/vision-model.git
%cd vision-model

# Install Python dependencies for segmentation and model downloading
!pip install -q transformers scipy torch torchvision huggingface_hub pillow

# Compile stable-diffusion.cpp with CUDA acceleration
!chmod +x setup.sh download_models.sh run_inference.sh edit_image.sh edit_segmented.sh
!./setup.sh

In [ ]:
# @title 3. Download Model Weights { run: "auto" }
# Choose between the Full Base Model (for 30-50 high-quality steps) or Turbo Model (4-step fast)
model_variant = "--base-q4 (Full Base Model, 30-50 steps)" # @param ["--base-q4 (Full Base Model, 30-50 steps)", "--turbo-q4 (Fast Viggle Turbo, 4 steps)", "--base-q8 (Near-Lossless, 30-50 steps)", "--turbo-q8"] {type:"string"}

flag = model_variant.split()[0]
print(f"Downloading weights with flag: {flag}...")
!./download_models.sh {flag}

In [ ]:
# @title 4. Text-to-Image Generation { run: "auto" }
prompt = "A raw 35mm photograph of an elderly craftsman carving wood in a sunlit workshop in Florence, photorealistic" # @param {type:"string"}
negative_prompt = "blurry, low quality, distorted, extra limbs, cartoon" # @param {type:"string"}
model_type = "Base Model (Qwen-Image-2.1-Q4_K_M.gguf)" # @param ["Base Model (Qwen-Image-2.1-Q4_K_M.gguf)", "Turbo Model (Qwen-Image-2.1-viggle-turbo-Q4_K_M.gguf)"]
steps = 35 # @param {type:"slider", min:1, max:50, step:1}
cfg_scale = 4.5 # @param {type:"slider", min:1.0, max:12.0, step:0.5}
width = 1024 # @param [512, 768, 1024, 1280] {type:"raw"}
height = 1024 # @param [512, 768, 1024, 1280] {type:"raw"}
seed = 42 # @param {type:"integer"}
output_file = "./output/colab_generation.png" # @param {type:"string"}

import os
from IPython.display import Image, display

if "Base" in model_type:
    os.environ["MODEL"] = "./models/diffusion_models/Qwen-Image-2.1-Q4_K_M.gguf"
else:
    os.environ["MODEL"] = "./models/diffusion_models/Qwen-Image-2.1-viggle-turbo-Q4_K_M.gguf"

os.environ["STEPS"] = str(steps)
os.environ["CFG"] = str(cfg_scale)
os.environ["WIDTH"] = str(width)
os.environ["HEIGHT"] = str(height)
os.environ["SEED"] = str(seed)
if negative_prompt.strip():
    os.environ["NEGATIVE_PROMPT"] = negative_prompt.strip()
else:
    os.environ.pop("NEGATIVE_PROMPT", None)

!./run_inference.sh "{prompt}" "{output_file}"

if os.path.exists(output_file):
    display(Image(output_file))


In [ ]:
# @title 5. Segmentation-Aware Pixel-Preserving Edit (High-Fidelity Base Model) { run: "auto" }
# Edits ONLY the target region, using 30-40 steps and negative prompts to resolve hand/tool anatomy
input_image = "assets/realistic_craftsman.png" # @param {type:"string"}
target_object = "the hand saw, wooden block, and hands" # @param {type:"string"}
edit_instruction = "An elderly baker in a clean kitchen cleanly slicing a gourmet chocolate cake with a chef knife, natural hands" # @param {type:"string"}
negative_prompt = "saw teeth, jagged blade, mangled hands, extra fingers, dirty table, wood shavings, cartoon, blurry" # @param {type:"string"}
model_variant = "Base Model (30-50 steps)" # @param ["Base Model (30-50 steps)", "Turbo Model (4 steps)"]
steps = 35 # @param {type:"slider", min:4, max:50, step:1}
cfg_scale = 4.5 # @param {type:"slider", min:1.0, max:10.0, step:0.5}
expand_pixels = 20 # @param {type:"slider", min:0, max:48, step:2}
feather_pixels = 4 # @param {type:"slider", min:0, max:16, step:1}
preview_masks_only = False # @param {type:"boolean"}
output_path = "./output/colab_segmented_cake.png" # @param {type:"string"}

import os
from IPython.display import Image, display

model_file = "./models/diffusion_models/Qwen-Image-2.1-Q4_K_M.gguf" if "Base" in model_variant else "./models/diffusion_models/Qwen-Image-2.1-viggle-turbo-Q4_K_M.gguf"

cmd = [
    "python", "edit_segmented.py",
    "--image", input_image,
    "--target", target_object,
    "--instruction", edit_instruction,
    "--output", output_path,
    "--expand-pixels", str(expand_pixels),
    "--feather-pixels", str(feather_pixels),
    "--steps", str(steps),
    "--cfg", str(cfg_scale),
    "--model", model_file,
]

if negative_prompt.strip():
    cmd.extend(["--negative-prompt", negative_prompt.strip()])

if preview_masks_only:
    cmd.append("--preview-masks")

import subprocess
print("Running Segmentation-Aware Editor on Colab GPU...")
proc = subprocess.run(cmd)

# Display results
base_name = os.path.splitext(os.path.basename(output_path))[0]
diag_dir = os.path.join(os.path.dirname(os.path.abspath(output_path)), f"{base_name}_diagnostics")
three_mask_panel = os.path.join(diag_dir, f"{base_name}_three_mask_panel.png")

if os.path.exists(three_mask_panel):
    print("\n--- Diagnostic Three-Mask Panel (Target, Allowed, Protected) ---")
    display(Image(three_mask_panel, width=800))

if os.path.exists(output_path) and not preview_masks_only:
    print("\n--- Final Pixel-Preserved Output ---")
    display(Image(output_path, width=800))


In [ ]:
# @title 6. Global Image-to-Image Edit (Full Canvas Regeneration) { run: "auto" }
input_image = "./output/colab_generation.png" # @param {type:"string"}
edit_prompt = "Replace the wooden block with a chocolate cake, and the saw with a chef knife" # @param {type:"string"}
negative_prompt = "saw teeth, mangled fingers, dirty table" # @param {type:"string"}
output_edit = "./output/colab_edit.png" # @param {type:"string"}
steps = 35 # @param {type:"slider", min:1, max:50, step:1}
cfg_scale = 4.5 # @param {type:"slider", min:1.0, max:15.0, step:0.5}
width = 1024 # @param [512, 768, 1024, 1280] {type:"raw"}
height = 1024 # @param [512, 768, 1024, 1280] {type:"raw"}

import os
from IPython.display import Image, display

os.environ["STEPS"] = str(steps)
os.environ["CFG"] = str(cfg_scale)
os.environ["WIDTH"] = str(width)
os.environ["HEIGHT"] = str(height)
if negative_prompt.strip():
    os.environ["NEGATIVE_PROMPT"] = negative_prompt.strip()
else:
    os.environ.pop("NEGATIVE_PROMPT", None)

!./edit_image.sh "{input_image}" "{edit_prompt}" "{output_edit}"

if os.path.exists(output_edit):
    display(Image(output_edit))
